# Последний эксперимент: высокие частоты

Сравниваем два режима на k ∈ [4,8]: (1) frozen модели, обученные на k ∈ [1,2], и (2) отдельное обучение всех трёх методов на новой полосе. Геометрия, DG0-среды и split IDs совпадают; test-частоты не участвуют ни в одном обучении. Высокочастотные решения вычисляем заново FEM-решателем. Это frequency shift, а не одновременная смена материалов/геометрии.

POD для второго режима строится заново только по high-band train. Для первого режима базисы и нормализация заморожены. Дополнительно проверяем h → h/2 на трёх test-частотах с максимальным k; число точек на длину волны само по себе не доказывает FEM-сходимость.

**Статичная задача:** геометрия и исходная среда №0 фиксированы; A,n и DG0-материалы одинаковы во всех примерах. Меняются k и θ. Один medium_id=0, разбиение по frequency_id: 670/170/160 частот (5360/1360/1280 полей). 1000 частот × 8 направлений = 8000 полей на полосу. Этот эксперимент проверяет интерполяцию параметров волны на одной среде.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from neural_operator.settings import (BASE, N_FREQUENCIES, N_DIRECTIONS, DATASET_TAG, POD_METHOD,
    LOW_DATA_DIR, HIGH_DATA_DIR, LOW_RUNS, HIGH_RUNS, SVD_RUNS, COMPARISON_RUNS)
DATA_DIR = LOW_DATA_DIR
RUNS = LOW_RUNS
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.data import load_dataset, write_dataset_info
from neural_operator.plotting import display_table, plot_prediction, plot_history


In [ ]:
from dataclasses import replace
from neural_operator.fem import StaticDatasetConfig, generate_static_dataset
from neural_operator.training import TrainConfig, run_experiment
from neural_operator.experiments import evaluate_frequency_shift, refinement_study
config_values = json.loads((DATA_DIR / 'config.json').read_text())
config_values.pop('medium_mode')
low_config = StaticDatasetConfig(**config_values)
high_config = replace(low_config, k_range=(4., 8.))
HIGH_DATA = HIGH_DATA_DIR
high = generate_static_dataset(HIGH_DATA, high_config)
high_info = write_dataset_info(high)
low_info = json.loads((DATA_DIR / 'dataset_info.json').read_text())
# Compare persisted array hashes without holding both large datasets in RAM.
for field in ['coordinates', 'triangles', 'cell_tags', 'A_cells', 'n_cells', 'theta', 'medium_id', 'frequency_id', 'split']:
    if high_info['field_sha256'][field] != low_info['field_sha256'][field]:
        raise ValueError(f'Low/high datasets differ in {field}')
print('High-band data:', high.u_scat.shape, 'k:', (high.k0.min(), high.k0.max()))


In [ ]:
RETRAIN = False
for method in ['svd', 'chebyshev', 'fno']:
    config = TrainConfig(method=method, epochs=200, width=24, depth=3,
                         rank=48, input_rank=32, resolution=32, modes=8, seed=42, threads=2, pod_method=POD_METHOD)
    _, metrics, _ = run_experiment(high, HIGH_RUNS / method, config, retrain=RETRAIN)
    print(method, metrics['test_fem_l2'])


In [ ]:
OUT = HIGH_RUNS / 'comparison'
low_dirs = {m: RUNS / m for m in ['svd', 'chebyshev', 'fno']}
high_dirs = {m: HIGH_RUNS / m for m in ['svd', 'chebyshev', 'fno']}
rows, summaries, resolution = evaluate_frequency_shift(low_dirs, high, OUT, high_dirs)
display_table(summaries)
display_table(resolution)


In [ ]:
refinement = refinement_study(high, OUT, n_groups=3)
display_table(refinement)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for regime, marker in [('frozen_low_band', 'o'), ('retrained_high_band', 'x')]:
    for method in ['SVD-SNO', 'Chebyshev-SNO', 'FNO']:
        selected = [r for r in rows if r['regime'] == regime and r['method'] == method]
        ax.scatter([r['k'] for r in selected], [r['error_fem_l2'] for r in selected],
                   marker=marker, alpha=.65, label=f'{method}, {regime}')
ax.set(yscale='log', xlabel='k', ylabel='Relative complex FEM L2')
ax.legend(fontsize=7, ncol=2, loc='upper center', bbox_to_anchor=(.5, -.20))
ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(OUT / 'frequency_errors.png', dpi=160)
plt.show()


Интерпретация: frozen error содержит frequency extrapolation и для SVD — ошибку переноса замороженного базиса. Retrained error проверяет другую обученную полосу при том же небольшом budget. Ошибка относительно fine FEM не учитывает всю ошибку FEM относительно непрерывной PDE; для неё отдельно смотрите refinement. Если rank/modes cap недостаточен, увеличение числа эпох не устраняет ограничение представления. Одна seed-серия на 1000 частотах — пилотное исследование, не доказательство высокочастотной точности.
